# Artwork Multi-Method Eval on Kaggle

This notebook runs artwork evaluation on Kaggle with three methods:

- `KVzip`
- `Finch_Full`
- `Finch_CPT`

It supports checkpoint resume, strict image checks, smoke tests, and CE-style evaluation output.

In [ ]:
import getpass
import os
import shutil
import socket
import subprocess
import sys
from pathlib import Path

BENCH_URL = "https://github.com/physical168/kv-compression-benchmark.git"
CE_URL = "https://github.com/GabrieleSanmartino/CompressionExperiments.git"

WORK_DIR = Path("/kaggle/working")
BENCH_DIR = WORK_DIR / "kv-compression-benchmark"
CE_DIR = WORK_DIR / "CompressionExperiments"

def load_github_token() -> str | None:
    # 1) Environment variable (recommended when set via Kaggle secrets as env)
    tok = (os.environ.get("GITHUB_TOKEN") or "").strip()
    if tok:
        return tok

    # 2) Kaggle UserSecretsClient
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        tok = (client.get_secret("GITHUB_TOKEN") or "").strip()
        if tok:
            return tok
    except Exception:
        pass

    # 3) Manual hidden input fallback
    try:
        manual = getpass.getpass("Paste GitHub token for private CE repo (leave empty to skip): ").strip()
        if manual:
            return manual
    except Exception:
        pass

    return None


gh_token = load_github_token()
print("GITHUB_TOKEN loaded:", bool(gh_token))

# Kaggle Internet switch must be ON, otherwise github.com cannot be resolved.
try:
    _ip = socket.gethostbyname("github.com")
    print("github.com resolved to:", _ip)
except Exception as e:
    raise RuntimeError(
        "Network/DNS is unavailable in this Kaggle session. "
        "Enable Notebook Settings -> Internet = ON, then restart and rerun. "
        f"Raw DNS error: {e}"
    )

def clone_public_or_token(url: str, out_dir: Path, token: str | None, repo_name: str) -> bool:
    r = subprocess.run(["git", "clone", "--depth", "1", url, str(out_dir)], capture_output=True, text=True)
    if r.returncode == 0:
        print(f"{repo_name}: public clone OK")
        return True
    print(f"{repo_name}: public clone failed (code={r.returncode})")
    if r.stderr:
        print((r.stderr or "")[:500])

    if not token:
        print(f"{repo_name}: token missing, skip token clone")
        return False

    private_url = url.replace("https://github.com/", f"https://{token}@github.com/")
    r2 = subprocess.run(["git", "clone", "--depth", "1", private_url, str(out_dir)], capture_output=True, text=True)
    if r2.returncode == 0:
        print(f"{repo_name}: token clone OK")
        return True
    print(f"{repo_name}: token clone failed (code={r2.returncode})")
    if r2.stderr:
        print((r2.stderr or "")[:500])
    return False

os.chdir(str(WORK_DIR))
for p in [BENCH_DIR, CE_DIR]:
    if p.exists():
        shutil.rmtree(p)

bench_ok = clone_public_or_token(BENCH_URL, BENCH_DIR, gh_token, "BENCH")
ce_ok = clone_public_or_token(CE_URL, CE_DIR, gh_token, "CE")
if not ce_ok:
    raise RuntimeError(
        "Failed to clone CompressionExperiments. "
        "If github.com is reachable, provide GITHUB_TOKEN with repo access to CE."
    )
print("Clone step finished. BENCH available:", bench_ok)

In [ ]:
import importlib
import inspect
import os
import subprocess
import sys
from pathlib import Path

os.chdir("/kaggle/working/CompressionExperiments")
subprocess.check_call(["git", "submodule", "update", "--init", "--recursive"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.45.0", "accelerate>=0.30.0", "bitsandbytes>=0.43.0",
    "pandas", "pyyaml", "scikit-learn", "matplotlib", "tqdm", "pillow"])

KV_REPO = Path("/kaggle/working/CompressionExperiments/kvpress").resolve()
# Kaggle 上可能已有残缺/命名空间的 kvpress；先清缓存再卸载后按绝对路径 editable 安装。
for _name in list(sys.modules):
    if _name == "kvpress" or _name.startswith("kvpress."):
        del sys.modules[_name]
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "kvpress"],
    capture_output=True,
    text=True,
)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(KV_REPO)])

# 关键：cwd 若在 CompressionExperiments，则 sys.path 里的 \"\" 会指向 CE 根目录；
# Python 会把 CE 下的子目录 kvpress/（子模块仓库根，通常没有顶层 __init__.py）当成 PEP420 命名空间，
# 从而 __file__==None 且没有 FinchPress。先离开该目录再 import。
os.chdir("/kaggle/working")
for _name in list(sys.modules):
    if _name == "kvpress" or _name.startswith("kvpress."):
        del sys.modules[_name]

# pip -e 在部分 Kaggle kernel 里不会注册到当前进程；强制把子模块根放在 sys.path 最前（包在 kvpress/kvpress/__init__.py）。
_inner_init = KV_REPO / "kvpress" / "__init__.py"
if not _inner_init.is_file():
    raise RuntimeError(f"kvpress 子模块不完整，缺少: {_inner_init}")
_p = str(KV_REPO)
while _p in sys.path:
    sys.path.remove(_p)
sys.path.insert(0, _p)

kvpress = importlib.import_module("kvpress")
_kvf = getattr(kvpress, "__file__", None)
if _kvf is None or not hasattr(kvpress, "FinchPress"):
    raise RuntimeError(
        "kvpress 仍未正确加载。"
        f" __file__={_kvf!r}, __path__={getattr(kvpress, '__path__', None)!r}. "
        "若 cwd 曾在 CE 目录，请先 chdir 到 /kaggle/working 再 import；或 Restart 后重跑本格。"
    )

# Optional file-level Finch hotfix. Some Kaggle runtimes install kvpress with different layout.
# If source file path is not available, runtime monkey patch in inference cell still applies.
try:
    from kvpress import FinchPress
    finch_path = Path(inspect.getfile(FinchPress))
    if finch_path.is_file():
        txt = finch_path.read_text(encoding="utf-8")
        txt_new = txt
        if "self.hook = None" not in txt_new and "def __enter__(self, model):" in txt_new:
            txt_new = txt_new.replace("def __enter__(self, model):", "def __enter__(self, model):\n        self.hook = None")
        txt_new = txt_new.replace("hook.remove()", "if hasattr(self, 'hook') and self.hook: self.hook.remove()")
        if txt_new != txt:
            finch_path.write_text(txt_new, encoding="utf-8")
            print("Applied Finch file hotfix:", finch_path)
        else:
            print("Finch file hotfix not needed.")
    else:
        print("Finch source path not found on disk, skip file hotfix:", finch_path)
except Exception as e:
    print("Skip Finch file hotfix:", str(e)[:200])

print("Install complete.")

In [ ]:
import os
import shutil
import time
import urllib.request
from pathlib import Path
from urllib.parse import unquote
import pandas as pd

SRC_BENCH_ART = Path("/kaggle/working/kv-compression-benchmark/datasets/artwork")
SRC_BENCH_CSV = SRC_BENCH_ART / "paintings.csv"
SRC_BENCH_IMG = SRC_BENCH_ART / "images"

DST_ART = Path("/kaggle/working/CompressionExperiments/experiment_manager/datasets/artwork")
DST_ART.mkdir(parents=True, exist_ok=True)
IMAGES_DIR = DST_ART / "images"
IMAGES_DIR.mkdir(exist_ok=True)
DATASET_PATH = DST_ART / "paintings.csv"

if SRC_BENCH_CSV.is_file():
    shutil.copy2(SRC_BENCH_CSV, DATASET_PATH)
    print("CSV source: benchmark repo")
elif DATASET_PATH.is_file():
    print("CSV source: CE built-in")
else:
    raise FileNotFoundError("No paintings.csv found")

# Prefer existing CE images, else copy benchmark images.
if IMAGES_DIR.is_dir() and any(IMAGES_DIR.iterdir()):
    print("Images source: CE built-in")
elif SRC_BENCH_IMG.is_dir() and any(SRC_BENCH_IMG.iterdir()):
    shutil.rmtree(IMAGES_DIR, ignore_errors=True)
    shutil.copytree(SRC_BENCH_IMG, IMAGES_DIR)
    print("Images source: benchmark repo")
else:
    raise FileNotFoundError("No usable images found")

# Robust downloader fallback: fill missing/invalid files for first rows.
def is_valid_img(p: Path) -> bool:
    return p.is_file() and p.stat().st_size > 1000

df_csv = pd.read_csv(DATASET_PATH)
MAX_TO_DOWNLOAD = 15
target_urls = df_csv["image_url"].dropna().head(MAX_TO_DOWNLOAD)
headers = {"User-Agent": "ArtworkBenchmarkBot/1.0 (contact: github.com/physical168)"}

print(f"Checking first {MAX_TO_DOWNLOAD} image files...")
for url in target_urls:
    fname_unquoted = unquote(url.split("/")[-1].split("?")[0])
    fname_quoted = url.split("/")[-1].split("?")[0]
    dst = IMAGES_DIR / fname_unquoted

    if is_valid_img(dst) or is_valid_img(IMAGES_DIR / fname_quoted):
        continue

    if dst.exists():
        dst.unlink()

    for attempt in range(3):
        try:
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=20) as r, open(dst, "wb") as f:
                f.write(r.read())
            print(f"Downloaded: {fname_unquoted}")
            time.sleep(1)
            break
        except Exception as e:
            if attempt == 2:
                print(f"Download failed: {fname_unquoted} -> {str(e)[:120]}")
            time.sleep(2)

print("Dataset ready:", DST_ART)
print("image count:", len(list(IMAGES_DIR.glob("*"))))

In [ ]:
import pandas as pd
from pathlib import Path
from urllib.parse import unquote
from PIL import Image, ImageFile

DATASET_PATH = Path("/kaggle/working/CompressionExperiments/experiment_manager/datasets/artwork/paintings.csv")
IMAGES_DIR = Path("/kaggle/working/CompressionExperiments/experiment_manager/datasets/artwork/images")
MAX_ROWS = 10

ImageFile.LOAD_TRUNCATED_IMAGES = False
df_check = pd.read_csv(DATASET_PATH).head(MAX_ROWS).copy()
bad_rows = []
for i, row in df_check.iterrows():
    tail = str(row["image_url"]).split("/")[-1].split("?")[0]
    p_un = IMAGES_DIR / unquote(tail)
    p_raw = IMAGES_DIR / tail
    target = p_un if p_un.is_file() else p_raw
    try:
        with Image.open(target) as im:
            _ = im.convert("RGB")
        print(f"OK Row {i}: {target.name}")
    except Exception as e:
        bad_rows.append((i, str(target), str(e)[:120]))
        print(f"BAD Row {i}: {target.name} -> {str(e)[:120]}")
if bad_rows:
    raise RuntimeError(f"Bad images found before model load: {bad_rows}")
print("Strict image check passed.")

In [ ]:
# Optional: targeted image repair by row id (run when strict check fails on specific rows)
import time
import urllib.request
from pathlib import Path
from urllib.parse import unquote
from PIL import Image, ImageFile
import pandas as pd

DATASET_PATH = Path("/kaggle/working/CompressionExperiments/experiment_manager/datasets/artwork/paintings.csv")
IMAGES_DIR = Path("/kaggle/working/CompressionExperiments/experiment_manager/datasets/artwork/images")

# Example: [1] for a single broken row. Keep empty to skip.
TARGET_RIDS = [1]
HEAD_ROWS = 10
MAX_RETRIES = 5
headers = {"User-Agent": "ArtworkBenchmarkBot/1.0 (contact: github.com/physical168)"}

if TARGET_RIDS:
    df_fix = pd.read_csv(DATASET_PATH).head(HEAD_ROWS).copy()
    for rid in TARGET_RIDS:
        if rid < 0 or rid >= len(df_fix):
            print(f"Skip rid={rid}: out of range for head({HEAD_ROWS})")
            continue

        row = df_fix.iloc[rid]
        url = str(row["image_url"])
        tail_raw = url.split("/")[-1].split("?")[0]
        tail_unq = unquote(tail_raw)

        p_raw = IMAGES_DIR / tail_raw
        p_unq = IMAGES_DIR / tail_unq

        # Remove both variants to avoid stale/corrupt copies.
        for p in [p_raw, p_unq]:
            if p.exists():
                p.unlink()

        dst = p_unq
        ok = False
        for attempt in range(MAX_RETRIES):
            try:
                req = urllib.request.Request(url, headers=headers)
                with urllib.request.urlopen(req, timeout=30) as r, open(dst, "wb") as f:
                    f.write(r.read())

                ImageFile.LOAD_TRUNCATED_IMAGES = False
                with Image.open(dst) as im:
                    _ = im.convert("RGB")

                print(f"[OK] repaired rid={rid}: {dst.name}, size={dst.stat().st_size}")
                ok = True
                break
            except Exception as e:
                print(f"[retry {attempt + 1}] rid={rid} failed: {str(e)[:160]}")
                time.sleep(2)

        if not ok:
            raise RuntimeError(f"Repair failed after retries for rid={rid}")
else:
    print("TARGET_RIDS is empty. Skip targeted repair.")

In [ ]:
import gc
import importlib.util
import os

import torch
from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration, BitsAndBytesConfig

# Reduce allocator fragmentation during large weight loads.
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

MODEL_ID = "llava-hf/llama3-llava-next-8b-hf"

if torch.cuda.is_available():
    torch.cuda.set_device(0)
    if "model" in globals():
        try:
            del model
        except NameError:
            pass
    gc.collect()
    torch.cuda.empty_cache()

compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

# 4-bit fits ~15GB T4 more reliably than 8-bit for this 8B VLM. Set USE_8BIT_MODEL = True to try 8-bit again.
USE_8BIT_MODEL = False
if USE_8BIT_MODEL:
    bnb_cfg = BitsAndBytesConfig(load_in_8bit=True, llm_int8_enable_fp32_cpu_offload=True)
else:
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

# Cap GPU 0 so a few layers can spill to CPU if needed (helps peak load memory).
max_memory = None
if torch.cuda.is_available():
    max_memory = {0: "12GiB", "cpu": "48GiB"}

processor = LlavaNextProcessor.from_pretrained(MODEL_ID)
model = LlavaNextForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=compute_dtype,
    device_map="auto",
    max_memory=max_memory,
    quantization_config=bnb_cfg,
)

_patch = "/kaggle/working/kv-compression-benchmark/benchmarks/artwork_eval/llava_kvpress_patch.py"
_spec = importlib.util.spec_from_file_location("patch", _patch)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
_mod.apply_kvpress_compatibility_patches(model)
print("Model ready. quant=", "8bit" if USE_8BIT_MODEL else "4bit", "max_memory=", max_memory)

In [ ]:
import os
import shutil
import yaml
import pandas as pd
from urllib.parse import unquote
from pathlib import Path
from PIL import Image, ImageFile
from kvpress import KVzipPress, FinchPress

MAX_ROWS = 10
MAX_NEW_TOKENS = 50
COMPRESSION_RATIOS = [0.4, 0.8, 0.95]
# FinchPress: tokens per clustering window (must be set; capped per prompt to sequence length).
FINCH_WINDOW_SIZE = 16

OUTPUT_ROOT = Path("/kaggle/working/ce_artwork_all_results")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_ROOT / "checkpoint_all.csv"
RESULTS_ROOT = OUTPUT_ROOT / "results"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# 跨会话续跑：working 会话结束会丢。把 checkpoint_all.csv 做成 Kaggle Dataset 并 Add data 后，按优先级：
# 1) 环境变量 CHECKPOINT_ALL_CSV=/kaggle/input/<你的数据集>/checkpoint_all.csv（Kaggle Secrets 里也可设同名 env）
# 2) /kaggle/input 下若恰好只有一个 checkpoint_all.csv，则自动选用
# 3) 否则在下面取消注释并写死 Path（或会话未断时无需导入，working 里已有文件）
CHECKPOINT_IMPORT_SOURCE = None
_env = (os.environ.get("CHECKPOINT_ALL_CSV") or "").strip()
if _env:
    CHECKPOINT_IMPORT_SOURCE = Path(_env)
else:
    _kin = Path("/kaggle/input")
    if _kin.is_dir():
        _cands = sorted(_kin.glob("**/checkpoint_all.csv"))
        if len(_cands) == 1:
            CHECKPOINT_IMPORT_SOURCE = _cands[0]
        elif len(_cands) > 1:
            print("WARN: 多个 checkpoint_all.csv，使用第一个。请设置 CHECKPOINT_ALL_CSV 或改代码指定路径:", _cands)
            CHECKPOINT_IMPORT_SOURCE = _cands[0]
# CHECKPOINT_IMPORT_SOURCE = Path("/kaggle/input/your-dataset-slug/checkpoint_all.csv")

if CHECKPOINT_IMPORT_SOURCE is not None and CHECKPOINT_IMPORT_SOURCE.is_file():
    shutil.copy2(CHECKPOINT_IMPORT_SOURCE, CHECKPOINT_PATH)
    print("Imported checkpoint:", CHECKPOINT_IMPORT_SOURCE, "->", CHECKPOINT_PATH)
elif CHECKPOINT_IMPORT_SOURCE is not None:
    print("WARN: CHECKPOINT_IMPORT_SOURCE 已设置但文件不存在 ->", CHECKPOINT_IMPORT_SOURCE)

EXPERIMENT_CONFIGS = {
    "KVzip": (KVzipPress, True),
    "Finch_Full": (FinchPress, False),
    "Finch_CPT": (FinchPress, True),
}

QCFG = Path("/kaggle/working/kv-compression-benchmark/benchmarks/artwork_eval/configs/image_queries.yaml")
with QCFG.open("r", encoding="utf-8") as f:
    q_cfg = yaml.safe_load(f)["artwork"]
queries_plan = [(q, True) for q in q_cfg["filter_queries"]] + [(q, False) for q in q_cfg["extract_queries"]]

df = pd.read_csv(DATASET_PATH).head(MAX_ROWS).copy()
df["record_id"] = range(len(df))
def resolve_image(url: str) -> str:
    tail = str(url).split("/")[-1].split("?")[0]
    p = IMAGES_DIR / unquote(tail)
    return str(p) if p.is_file() else str(IMAGES_DIR / tail)
df["image_path"] = df["image_url"].apply(resolve_image)

# Pre-scan bad images to avoid repeated failures in full loop.
ImageFile.LOAD_TRUNCATED_IMAGES = True
BAD_RIDS = set()
for _, row in df.iterrows():
    rid = int(row["record_id"])
    p = row["image_path"]
    try:
        with Image.open(p) as im:
            im.verify()
    except Exception as e:
        BAD_RIDS.add(rid)
        print(f"Bad image detected: rid={rid}, path={p}, err={str(e)[:120]}")

def apply_finch_hook_fix():
    """Upstream FinchPress.__call__ uses hook.remove() in finally without initializing hook.
    If register_forward_hook fails, Python raises 'cannot access local variable hook'.
    Replace __call__ with a safe version (hook=None + guarded remove)."""
    if getattr(FinchPress, "_hook_fix_applied", False):
        return

    from contextlib import contextmanager

    @contextmanager
    def __call__(self, model):
        if self.delimiter_token_id is None:
            raise ValueError(
                "No delimiter token ID provided. Use update_model_and_tokenizer before calling the press."
            )
        self._is_multimodal = hasattr(model, "language_model")
        embed_tokens = (
            model.language_model.embed_tokens
            if self._is_multimodal
            else model.model.embed_tokens
        )
        with super(FinchPress, self).__call__(model):
            hook = None
            try:
                hook = embed_tokens.register_forward_hook(self.embed_token_forward_hook)
                yield
            finally:
                if hook is not None:
                    hook.remove()

    FinchPress.__call__ = __call__
    FinchPress._hook_fix_applied = True
    print("Applied FinchPress.__call__ hook fix.")

apply_finch_hook_fix()


def set_finch_window_for_inputs(press, input_ids) -> None:
    """kvpress FinchPress raises if window_size is unset; it must not exceed prompt length."""
    if not isinstance(press, FinchPress):
        return
    seq_len = int(input_ids.shape[1])
    press.window_size = max(1, min(int(FINCH_WINDOW_SIZE), seq_len))

_EXTRACT_SUFFIX = (" (be concise, no explanation, no introductory text, just the answer,"
                   " output datatype: STRING, do not repeat the datatype in the answer) ?")
def build_answer_prefix(row, question: str, is_boolean: bool, use_cpt: bool) -> str:
    context = "" if use_cpt else f"This painting is titled '{row['title']}', created around {row['inception']}. It belongs to the {row['movement']} movement and {row['genre']} genre. "
    if is_boolean:
        return (f"{context}Answer the following question based on the image with '1' or '0'. Do not add any other comments. {question}\nAnswer: ")
    return (f"{context}Answer the following question based on the image. Do not add any other comments. {question + _EXTRACT_SUFFIX}\nAnswer: ")

print(f"Configuration ready. BAD_RIDS={sorted(BAD_RIDS)}")

In [ ]:
import time
import torch
from contextlib import contextmanager
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = False

if "apply_finch_hook_fix" in globals():
    apply_finch_hook_fix()

SMOKE_ROWS = min(3, len(df))
SMOKE_METHODS = ["KVzip", "Finch_Full", "Finch_CPT"]
SMOKE_RATIO = 0.4
SMOKE_MAX_NEW_TOKENS = 12
qtext, is_bool = queries_plan[0]
_tok = getattr(processor, "tokenizer", processor)

@contextmanager
def safe_press_context(press, model):
    cm = press(model)
    cm.__enter__()
    try:
        yield
    finally:
        try:
            cm.__exit__(None, None, None)
        except UnboundLocalError:
            # Workaround for Finch hook bug in some kvpress builds.
            pass

def _run_once(image_path, answer_prefix, PressCls, ratio):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    press = PressCls(compression_ratio=float(ratio))
    if hasattr(press, "update_model_and_tokenizer"):
        press.update_model_and_tokenizer(model, _tok)
    image = Image.open(image_path).convert("RGB")
    conv = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": answer_prefix}]}]
    prompt = processor.apply_chat_template(conv, add_generation_prompt=True)
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device)
    set_finch_window_for_inputs(press, inputs.input_ids)
    with torch.no_grad(), safe_press_context(press, model):
        out = model.generate(**inputs, max_new_tokens=SMOKE_MAX_NEW_TOKENS, pad_token_id=_tok.pad_token_id)
    return _tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

results = []
for i in range(SMOKE_ROWS):
    row = df.iloc[i]
    rid = int(row["record_id"])
    img_path = row["image_path"]
    with Image.open(img_path) as im:
        _ = im.convert("RGB")
    for m_name in SMOKE_METHODS:
        Cls, use_cpt = EXPERIMENT_CONFIGS[m_name]
        ap = build_answer_prefix(row, qtext, is_bool, use_cpt)
        t0 = time.perf_counter()
        ok, err, out = True, "", ""
        try:
            out = _run_once(img_path, ap, Cls, SMOKE_RATIO)
        except Exception as e:
            ok = False
            err = str(e)[:200]
        dt = (time.perf_counter() - t0) * 1000
        results.append((rid, m_name, ok, dt, out[:100], err))

for rid, m_name, ok, dt, out_head, err_head in results:
    if ok:
        print(f"PASS rid={rid} method={m_name} t={dt:.1f}ms out={out_head!r}")
    else:
        print(f"FAIL rid={rid} method={m_name} t={dt:.1f}ms err={err_head!r}")
if any(not r[2] for r in results):
    raise RuntimeError("Smoke test failed. Fix before full run.")
print("Smoke test passed.")

In [ ]:
import csv as _csv
import gc
import logging
import os
import warnings
from contextlib import contextmanager
import pandas as pd
import torch
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True
logging.getLogger("kvpress.presses.kvzip_press").setLevel(logging.ERROR)
logging.getLogger("kvpress.presses.base_press").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*MatMul8bitLt.*")

@contextmanager
def safe_press_context(press, model):
    cm = press(model)
    cm.__enter__()
    try:
        yield
    finally:
        try:
            cm.__exit__(None, None, None)
        except UnboundLocalError:
            # Workaround for Finch hook bug in some kvpress builds.
            pass

# Re-apply Finch hook fix if cells were run out of order.
if "apply_finch_hook_fix" in globals():
    apply_finch_hook_fix()

_tok = getattr(processor, "tokenizer", processor)
def run_generate_vision(image_path, answer_prefix, PressCls, ratio):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    press = PressCls(compression_ratio=float(ratio))
    if hasattr(press, "update_model_and_tokenizer"):
        press.update_model_and_tokenizer(model, _tok)
    image = Image.open(image_path).convert("RGB")
    conv = [{"role":"user","content":[{"type":"image"},{"type":"text","text":answer_prefix}]}]
    prompt = processor.apply_chat_template(conv, add_generation_prompt=True)
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device)
    set_finch_window_for_inputs(press, inputs.input_ids)
    with torch.no_grad(), safe_press_context(press, model):
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, pad_token_id=_tok.pad_token_id)
    return _tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def load_done(path):
    if not path.exists():
        return set()
    ck = pd.read_csv(path)
    done = set()
    for _, r in ck.iterrows():
        if str(r.get("error", "")).strip().lower() in ("", "nan"):
            done.add((str(r["method"]), str(float(r["ratio"])), int(r["record_id"]), str(r["query"])))
    return done

done_keys = load_done(CHECKPOINT_PATH)
print(f"Resuming: {len(done_keys)} already done.")

f = CHECKPOINT_PATH.open("a", newline="", encoding="utf-8")
writer = _csv.DictWriter(f, fieldnames=["method", "ratio", "record_id", "query", "answer", "error"])
if CHECKPOINT_PATH.stat().st_size == 0:
    writer.writeheader()

total_tasks = len(EXPERIMENT_CONFIGS) * len(COMPRESSION_RATIOS) * len(df) * len(queries_plan)
n_step = 0
n_err = 0

for m_name, (Cls, use_cpt) in EXPERIMENT_CONFIGS.items():
    print(f"\n>>> Starting Method: {m_name}")
    for ratio in COMPRESSION_RATIOS:
        for _, row in df.iterrows():
            rid = int(row["record_id"])
            if rid in globals().get("BAD_RIDS", set()):
                continue
            for qtext, is_bool in queries_plan:
                n_step += 1
                key = (m_name, str(float(ratio)), rid, qtext)
                if key in done_keys:
                    continue

                ap = build_answer_prefix(row, qtext, is_bool, use_cpt)
                ans = ""
                err = ""
                try:
                    ans = run_generate_vision(row["image_path"], ap, Cls, ratio)
                except Exception as e:
                    err = str(e)[:500]
                    n_err += 1
                    print(f"ERROR [rid={rid} {m_name} R{ratio}]: {err[:120]}")

                writer.writerow({"method": m_name, "ratio": ratio, "record_id": rid, "query": qtext, "answer": ans, "error": err})
                f.flush()

                if n_step % 20 == 0:
                    print(f"Progress: {n_step}/{total_tasks} | Errors so far: {n_err}")

                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

f.close()
print(f"Inference finished. Total errors: {n_err}")

In [ ]:
import pandas as pd
import sys
from pathlib import Path

ck_all = pd.read_csv(CHECKPOINT_PATH)
ea_path = Path("/kaggle/working/evaluation_results.csv")

ok = ck_all[ck_all.error.fillna("").astype(str).str.strip().isin(["", "nan"])].copy()
for (m, r), grp in ok.groupby(["method", "ratio"]):
    d = RESULTS_ROOT / "artwork" / f"Llava3-8b-{m}" / f"{float(r):.2f}"
    d.mkdir(parents=True, exist_ok=True)
    grp[["record_id", "query", "answer"]].to_csv(d / "results.csv", index=False)

sys.path.append("/kaggle/working/kv-compression-benchmark/benchmarks/artwork_eval")
from evaluation.evaluator import EvaluationManager
mgr = EvaluationManager(
    config_path="/kaggle/working/kv-compression-benchmark/benchmarks/artwork_eval/evaluation/evaluation_config.yaml",
    results_dir=str(RESULTS_ROOT),
)
res = mgr.evaluate_all()
out_csv = OUTPUT_ROOT / "evaluation_results_all_methods.csv"
res.to_csv(out_csv, index=False)
print("Wrote:", out_csv)
display(res.groupby(["model_tag", "ratio"])[["precision", "recall", "f1"]].mean())